In [ ]:
from dotenv import load_dotenv
import os
load_dotenv()                              # reads .env file into environment
api_key = os.getenv("ANTHROPIC_API_KEY")   # retrieves the key string
print("API key loaded:", api_key is not None)

In [ ]:
from anthropic import Anthropic
client = Anthropic(api_key=api_key)   # creates a reusable API client
print("Anthropic client ready")

In [ ]:
response = client.messages.create(
    model="claude-haiku-4-5",          # fast, cheap model for smoke test
    #    model="claude-sonnet-4-5",
    max_tokens=50,                     # hard ceiling on output tokens
    messages=[{"role": "user", "content": "Say hello in one short sentence."}]
)
print(response.content[0].text)        # content is a list; [0] is the first block

# CCA Lab: Prompt Engineering & Evaluation Fundamentals

**Course:** Anthropic Academy — Claude API Foundations  
**Sub-module:** Prompt Design · Evaluation · Improvement Cycles  
**Lesson:** End-to-end prompt engineering with structured rubrics

## What this lab covers
- Calling the Messages API with system prompts and multi-turn conversations
- Building a numeric scoring rubric with per-criterion output
- Running a full write → evaluate → improve → re-evaluate cycle
- Diagnosing failure modes with live code demos
- Tracking token usage across every API call

## CCA Domains Exercised
Prompt construction · Evaluation design · Iterative improvement · Error handling · Token accounting

## Learning Objectives
1. Explain every parameter in a `messages.create()` call
2. Write a rubric function that prints per-dimension pass/fail scores
3. Demonstrate a three-turn conversation with visible message-list growth
4. Identify and trigger at least one common API failure mode
5. Produce a cumulative token-usage table across multiple calls

## Section 1: Prerequisites & API Glossary
### CCA objective demonstrated: Locate and explain every Messages API parameter

| Parameter | Type | Required | Description |
|-----------|------|----------|-------------|
| `model` | string | ✅ | Model ID — controls capability vs cost trade-off |
| `max_tokens` | int | ✅ | Hard ceiling on output tokens; prevents runaway cost |
| `messages` | list[dict] | ✅ | Alternating `user`/`assistant` turns — the conversation |
| `system` | string | ❌ | Persistent instruction layer injected before the conversation |
| `temperature` | float 0-1 | ❌ | Randomness; 0 = deterministic, 1 = creative |
| `stop_sequences` | list[str] | ❌ | Tokens that halt generation early |

**Install deps:** `pip install anthropic python-dotenv`  
**`.env` file:** `ANTHROPIC_API_KEY=sk-ant-…`

## Section 2: The `ask()` Helper & System Prompts
### CCA objective demonstrated: Build a reusable helper; explain system prompt architecture

### System parameter — when and why

| What belongs in `system` | What belongs in the `user` turn |
|--------------------------|----------------------------------|
| Persistent persona / role | The specific task or question |
| Output format rules (JSON, markdown) | Dynamic context or data |
| Constraints that never change | One-off instructions |
| Domain knowledge scope | User-supplied variables |

**Architectural principle:** Put everything that is constant across all turns in `system`; put everything that varies turn-by-turn in `messages`.

> **Grader reliability note:** Deterministic rubrics can over-score responses that
> contain the right words but wrong semantics. For production evals, complement
> keyword checks with a Claude-as-judge semantic pass — the rule + judge layering
> pattern from the evaluation labs.

In [ ]:
usage_log = []   # accumulates token counts for every API call this session

def ask(prompt, system="", model="claude-haiku-4-5", max_tokens=512, temperature=0):
    """
    Send a single-turn message and return the assistant reply string.

    Parameters
    ----------
    prompt      : str   — the user message text
    system      : str   — persistent instructions (optional)
    model       : str   — Anthropic model ID
    max_tokens  : int   — hard ceiling on response length
    temperature : float — 0 = deterministic, useful for evals

    Returns
    -------
    str — the assistant's reply
    """
    kwargs = dict(
        model=model,                                  # which model to call
        max_tokens=max_tokens,                        # never exceed this token count
        temperature=temperature,                      # reproducibility for evals
        messages=[{"role": "user", "content": prompt}]  # minimal single-turn list
    )
    if system:                          # only attach system if caller provided one
        kwargs["system"] = system       # keyword arg — API docs specify this key

    resp = client.messages.create(**kwargs)  # make the HTTP request

    # resp.content is a list of content blocks (text, tool_use, etc.)
    # We index [0] because we expect exactly one TextBlock in a plain chat reply
    reply = resp.content[0].text

    # resp.stop_reason tells us WHY generation stopped:
    #   "end_turn"   = model finished naturally
    #   "max_tokens" = hit our ceiling — response may be truncated!
    stop = resp.stop_reason

    usage_log.append({                          # record for token analysis later
        "label": prompt[:40],                   # first 40 chars as identifier
        "input": resp.usage.input_tokens,       # tokens sent to the model
        "output": resp.usage.output_tokens,     # tokens the model produced
        "stop": stop,
    })
    return reply

# Smoke test with system prompt
reply = ask(
    "What is your role?",
    system="You are a concise Python tutor. Reply in one sentence."
)
print("Reply:", reply)

## Section 3: Intentional Error Demo & Multi-Turn Conversations
### CCA objective demonstrated: Read API errors; build multi-turn message lists

In [ ]:
# --- Intentional error: omit max_tokens ---
# This triggers a validation error so students learn to read API errors early.
from anthropic import BadRequestError

try:
    bad = client.messages.create(
        model="claude-haiku-4-5",
        # max_tokens intentionally omitted — required field!
        messages=[{"role": "user", "content": "Hello"}]
    )
except Exception as e:
    print(f"Caught error type : {type(e).__name__}")
    print(f"Error message     : {e}")
    print("\n✅ Expected — max_tokens is required by the API.")

In [ ]:
# Multi-turn conversation — messages list grows with every turn
messages = []   # start with empty history

def chat(user_text, system=""):
    """Append user turn, call API, append assistant reply, return reply."""
    messages.append({"role": "user", "content": user_text})  # add user turn

    kwargs = dict(model="claude-haiku-4-5", max_tokens=256,
                  temperature=0, messages=messages)
    if system:
        kwargs["system"] = system

    resp = client.messages.create(**kwargs)
    reply = resp.content[0].text

    messages.append({"role": "assistant", "content": reply})  # add assistant turn

    usage_log.append({"label": f"chat-turn-{len(messages)//2}",
                      "input": resp.usage.input_tokens,
                      "output": resp.usage.output_tokens,
                      "stop": resp.stop_reason})
    return reply

SYS = "You are a helpful Python tutor. Be brief."

# Turn 1
chat("What is a list comprehension?", system=SYS)
print("=== Messages after Turn 1 ===")
for m in messages:
    print(f"  [{m['role']}] {m['content'][:80]}")

# Turn 2
chat("Give me a one-line example.", system=SYS)
print("\n=== Messages after Turn 2 ===")
for m in messages:
    print(f"  [{m['role']}] {m['content'][:80]}")

# Turn 3
chat("How does that compare to a regular for-loop?", system=SYS)
print("\n=== Messages after Turn 3 ===")
for m in messages:
    print(f"  [{m['role']}] {m['content'][:80]}")

## Section 4: Rubric Scoring
### CCA objective demonstrated: Operationalise quality criteria into numeric scores with per-dimension output

In [ ]:
import re

def score_explanation(text):
    """
    Score a Python-concept explanation on four dimensions (0-3 each, max 12).

    Returns
    -------
    int — total score 0-12
    """
    total = 0

    # Dimension 1 — Clarity: contains 'is' or 'means' or 'allows'
    # re.search scans the whole string (vs re.match which only checks the start)
    # \b ensures we match whole words, not fragments like 'this'
    d1 = 3 if re.search(r'\b(is|means|allows)\b', text, re.IGNORECASE) else 0
    icon1 = "✅" if d1 else "❌"
    print(f"  {icon1} Clarity (definition word)     : {d1}/3")
    total += d1

    # Dimension 2 — Example: contains a code block marker or 'example'
    d2 = 3 if re.search(r'(```|example|for instance)', text, re.IGNORECASE) else 0
    icon2 = "✅" if d2 else "❌"
    print(f"  {icon2} Example present               : {d2}/3")
    total += d2

    # Dimension 3 — Conciseness: under 300 characters
    d3 = 3 if len(text) < 300 else 0
    icon3 = "✅" if d3 else "❌"
    print(f"  {icon3} Concise (<300 chars, got {len(text)}) : {d3}/3")
    total += d3

    # Dimension 4 — Beginner language: avoids jargon words
    jargon = ["polymorphism", "metaclass", "dunder", "mixin"]
    # List comprehension: for each word in jargon, check if it's in the text
    hits = [w for w in jargon if w in text.lower()]
    d4 = 3 if not hits else 0
    icon4 = "✅" if d4 else "❌"
    print(f"  {icon4} No jargon (hits={hits})        : {d4}/3")
    total += d4

    print(f"  {'─'*40}")
    print(f"  TOTAL SCORE: {total}/12")
    return total

# Test on a sample response
sample = ask("Explain what a Python list is. Keep it under 100 words.")
print("=== Rubric for sample response ===")
s = score_explanation(sample)
print(f"\nResponse (first 120 chars): {sample[:120]}")

## Section 5: Improvement Cycle — Write → Evaluate → Improve → Re-evaluate
### CCA objective demonstrated: Iterative prompt refinement with tracked numeric progress

> **Grader reliability note:** Deterministic rubrics can over-score responses that
> contain the right words but wrong semantics. For production evals, complement
> keyword checks with a Claude-as-judge semantic pass — the rule + judge layering
> pattern from the evaluation labs.

In [ ]:
THRESHOLD = 9   # score needed to PASS (out of 12)

versions = [
    {"label": "v1-bare",
     "prompt": "Explain Python dictionaries."},
    {"label": "v2-length",
     "prompt": "Explain Python dictionaries in under 60 words."},
    {"label": "v3-full",
     "prompt": ("Explain Python dictionaries in under 60 words. "
                "Include one short code example. Use simple language.")},
]

results = []
for v in versions:
    print(f"\n{'='*50}")
    print(f"Prompt version : {v['label']}")
    print(f"Prompt text    : {v['prompt'][:70]}")
    resp_text = ask(v["prompt"], temperature=0)
    print("Scoring:")
    sc = score_explanation(resp_text)
    results.append({"label": v["label"], "prompt": v["prompt"], "score": sc})

# Side-by-side comparison table
print("\n" + "="*60)
print(f"{'Version':<14} {'Score':>6}  {'Prompt (truncated)':<35}")
print("-"*60)
for r in results:
    flag = "✅ PASS" if r["score"] >= THRESHOLD else "❌ FAIL"
    print(f"{r['label']:<14} {r['score']:>5}/12  {r['prompt'][:35]:<35}  {flag}")

best = max(results, key=lambda x: x["score"])
print("\n" + ("✅ IMPROVEMENT CYCLE PASSED" if best["score"] >= THRESHOLD
              else "❌ IMPROVEMENT CYCLE FAILED — refine prompts further"))

## Section 6: Failure Mode Analysis
### CCA objective demonstrated: Anticipate, document, and handle common API failure modes

| Failure Mode | Trigger | Symptom |
|---|---|---|
| Missing `max_tokens` | Omit required parameter | `BadRequestError` / `ValidationError` from the API |
| `max_tokens` too small | Set `max_tokens=5` for long output | `stop_reason == "max_tokens"` — response cut off mid-sentence |
| Invalid model ID | Typo in model string | `NotFoundError` — model does not exist |
| Wrong message role | Use `"system"` inside messages list | `BadRequestError` — role must be user or assistant |
| Empty messages list | Pass `messages=[]` | `BadRequestError` — at least one message required |

In [ ]:
from anthropic import APIStatusError

# Failure 1: max_tokens truncation
print("--- Failure 1: max_tokens too small ---")
trunc = client.messages.create(
    model="claude-haiku-4-5",
    max_tokens=5,              # deliberately tiny
    messages=[{"role": "user", "content": "Write a 3-sentence summary of Python."}]
)
usage_log.append({"label": "failure-truncation",
                  "input": trunc.usage.input_tokens,
                  "output": trunc.usage.output_tokens,
                  "stop": trunc.stop_reason})
print(f"stop_reason : {trunc.stop_reason}")
print(f"reply text  : {trunc.content[0].text!r}")
if trunc.stop_reason == "max_tokens":
    print("⚠️  Response was truncated — increase max_tokens!")

# Failure 2: invalid model ID
print("\n--- Failure 2: invalid model ID ---")
try:
    client.messages.create(
        model="claude-does-not-exist-9999",
        max_tokens=10,
        messages=[{"role": "user", "content": "Hi"}]
    )
except APIStatusError as e:
    print(f"Caught {type(e).__name__}: status={e.status_code}")
    print("✅ Expected — model ID must be a valid Anthropic model string.")

# Failure 3: wrong role in messages
print("\n--- Failure 3: 'system' role inside messages list ---")
try:
    client.messages.create(
        model="claude-haiku-4-5",
        max_tokens=10,
        messages=[{"role": "system", "content": "Be helpful."},
                  {"role": "user",   "content": "Hi"}]
    )
except Exception as e:
    print(f"Caught {type(e).__name__}: {str(e)[:120]}")
    print("✅ Expected — use the top-level 'system' parameter, not messages list.")

## Section 7: Token Usage Analysis
### CCA objective demonstrated: Track and interpret cumulative token costs across a session

In [ ]:
# Print a per-call usage table with running cumulative totals
print(f"{'#':<4} {'Label':<35} {'Input':>7} {'Output':>7} {'CumIn':>7} {'CumOut':>7}")
print("-" * 68)

cum_in = 0
cum_out = 0
for i, entry in enumerate(usage_log, 1):
    cum_in  += entry["input"]   # running total of input tokens
    cum_out += entry["output"]  # running total of output tokens
    label = entry["label"][:34]  # truncate long labels
    print(f"{i:<4} {label:<35} {entry['input']:>7} {entry['output']:>7} {cum_in:>7} {cum_out:>7}")

print("-" * 68)
print(f"{'TOTAL':<40} {cum_in:>7} {cum_out:>7}")
print(f"\nSession grand total: {cum_in + cum_out} tokens across {len(usage_log)} API calls")

## Section 8: Practice Drills
### CCA objective demonstrated: Apply all concepts independently

Complete the three exercises below. Each builds on skills from this lab.

In [ ]:
# ── Drill 1 ──────────────────────────────────────────────────────────────
# Write a system prompt that makes Claude respond ONLY in bullet points.
# Then call ask() and verify the output starts with a bullet.

drill1_system = "Always respond using bullet points only. Never use prose paragraphs."
drill1_reply  = ask("List three benefits of Python.", system=drill1_system)
starts_bullet = drill1_reply.strip().startswith(("•", "-", "*", "1."))
print("Drill 1 — bullet output:", "✅ PASS" if starts_bullet else "❌ FAIL")
print(drill1_reply[:200])

# ── Drill 2 ──────────────────────────────────────────────────────────────
# Add a 5th rubric dimension: checks that the word 'example' appears.
# Score it and print the ✅/❌ icon.

def score_with_d5(text):
    """Extend score_explanation with a 5th dimension: explicit 'example' word."""
    base = score_explanation(text)
    d5 = 3 if "example" in text.lower() else 0
    icon = "✅" if d5 else "❌"
    print(f"  {icon} Explicit 'example' keyword    : {d5}/3")
    return base + d5

print("\nDrill 2 — extended rubric:")
d2_text = ask("What is a Python tuple? Give an example in under 60 words.")
score_with_d5(d2_text)

# ── Drill 3 ──────────────────────────────────────────────────────────────
# Run a 2-turn conversation asking about a Python concept, then ask for
# a quiz question on the same concept. Print both full messages lists.

drill3_msgs = []
def chat3(user_text):
    """Minimal multi-turn helper for Drill 3."""
    drill3_msgs.append({"role": "user", "content": user_text})
    r = client.messages.create(model="claude-haiku-4-5", max_tokens=200,
                                messages=drill3_msgs)
    drill3_msgs.append({"role": "assistant", "content": r.content[0].text})
    usage_log.append({"label": f"drill3-{len(drill3_msgs)//2}",
                      "input": r.usage.input_tokens,
                      "output": r.usage.output_tokens, "stop": r.stop_reason})

chat3("In one sentence, what is a Python set?")
chat3("Now give me one quiz question about sets.")
print("\nDrill 3 — full messages list after 2 turns:")
for m in drill3_msgs:
    print(f"  [{m['role']}] {m['content'][:90]}")

## Section 9: CCA Takeaways
### CCA objective demonstrated: Consolidate exam-ready mental models

| # | Mental Model | Exam Tip |
|---|---|---|
| 1 | **`system` vs `messages`** | Static instructions → `system`; dynamic content → `messages` |
| 2 | **`max_tokens` is a ceiling** | Always set it; `stop_reason=="max_tokens"` means truncation |
| 3 | **`response.content` is a list** | Index `[0]` for the first (usually only) TextBlock |
| 4 | **Rubric = deterministic first pass** | Keyword scoring is fast but blind to semantics — layer a judge on top |
| 5 | **Token log every call** | Append `input_tokens` + `output_tokens` after every `messages.create()` to track cost |

---

**Lab complete ✅** — Run *Kernel → Restart and Run All Cells* to verify end-to-end execution.